In [0]:
jdbc_url = "jdbc:sqlserver://sql-employee-dev.database.windows.net:1433;database=datawarehousedb"

connection_properties = {
    "user": "sql",
    "password": "Dpkr@123",
    "driver": "com.microsoft.sqlserver.jdbc.SQLServerDriver"
}

In [0]:
gold_df = spark.read.table("adw2.default.gold_employee_summary")

display(gold_df)

In [0]:
import time

max_retries = 5
retry_delay = 30  # seconds between retries (allows Azure SQL Serverless to resume)

for attempt in range(1, max_retries + 1):
    try:
        gold_df.write.jdbc(
            url=jdbc_url,
            table="gold_employee_summary",
            mode="overwrite",
            properties=connection_properties
        )
        print(f"Successfully wrote gold_employee_summary to Azure SQL (attempt {attempt}).")
        break
    except Exception as e:
        print(f"Attempt {attempt}/{max_retries} failed: {e}")
        if attempt < max_retries:
            print(f"Retrying in {retry_delay}s (database may be resuming from auto-pause)...")
            time.sleep(retry_delay)
        else:
            raise

In [0]:
%sql
SELECT * FROM gold_employee_summary;